# 1.09 - Named Entity Recognition

This notebook analyzes named entities extracted [SpaCY](https://spacy.io/).

## **Named_Entities**


### **Format**
  - list [(entity, label), ...]
  - Default Value : []

  - Named Entities : 
      - CARDINAL
      - DATE
      - EVENT
      - FAC
      - GPE
      - LANGUAGE
      - LAW
      - LOC
      - MONEY
      - NORP
      - ORDINAL
      - ORG
      - PERCENT
      - PERSON
      - PRODUCT
      - QUANTITY
      - TIME
      - WORK_OF_ART
 
### **Method**

Used [SpaCYs en_core_web_trf](https://spacy.io/models/en#en_core_web_trf).
  - trained on written blogs, news, and comments
  - Highest F1 score of pre-trained models (0.90)
  - Disabled unnecessary modules to speed up computation
      - parser
      - attribute_ruler
      - lemmatizer

### **For Graders**

There are 4 major sections:
  - **Feature Extraction** 
    - extract named entities
    - count occurence of each named entity using `collections.Counter`
  - **SpaCY Performance** 
    - Coverage Analysis
    - entity label distribution
    - Highlight cool entities  
  - **Improving on Assignment 1** 
    - Improve `extract_dates` and `classify_time_of_day` from A1 using named entities as edge cases.
    - Document changes made to functions and coverage improvements.
  - **Regenerated A1 Features**
    - rerun A1 scripts with new and improved functions. Table of improved performance below :point_down:
    
|    |   Time_of_Day |   Haunted_Places_Date |
|:---|--------------:|----------------------:|
| v1 |       32.84\% |               25.48\% |
| v2 |       **35.34**\% |               **27.57**\% |


**Final Note**:
  - I looked into using tools like [label_sudio](https://labelstud.io/) to curate a training set and fine tune our model to haunted places. However, in order to parse for custom entities like ("GHOST", "DEMON", etc.), we would need an annotated dataset of 1,000s of entities of each type. **We would have to train on almost the entire haunted places dataset**. 



    

### **Imports and Data**

In [ ]:
# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Pandas and plots#
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt

import json
import re
import numpy as np

# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import ast
from typing import Pattern
from itertools import chain
from collections import Counter

# Dates #
import datetime
import datefinder
from dsci_550_a1.parsingFunctions import extract_dates, clean_dates

# SpaCY # 
import spacy as sp


# Data #

# Input TSV #
df = pd.read_csv("../data/processed/haunted_places_features_added_v2.tab", index_col = "Haunted_Places_Id", sep = "\t")

# Outfile TSV #
outfile = "../data/processed/haunted_places_features_added_v2.tab"

feature = "Named_Entities"

# convert feature column from string to list if its in df
if feature in df.columns:
    df[feature] = df[feature].apply(ast.literal_eval) 

nlp = sp.load("en_core_web_trf" , disable= [
    "parser",
    "attribute_ruler", 
    "lemmatizer"
])

# Display possible labels
ner_labels = nlp.get_pipe("ner").labels
print("Available labels: ", ner_labels)

### **Functions**

In [ ]:
#########################################################
## Entity extraction functions ##

def process_docs(docs): 
    '''
    Extract named entities from a list of documents

    Input:
        [docs]                  | list of documents

    Returns:
        [ent_data]              | list of entities extracted from a list of documents
                                | format:   [[entity_1, ner_label_1], [entity_2, ner_label_2], ...]

        [ent_counts]            | dictionary of counter objects.
                                | format:   {ner_label : {entity_1 : count, 
                                                          entity_2 : count}, ...
                                                         },
                                            ner_label_2 : {...}, ...
                                            }                                     
    '''
    # initialize list to store extracted entities
    ent_data = []

    # intialize entity counter dictionary
    ent_counts = {label : Counter() for label in ner_labels}

    # iterate though each document
    for doc in tqdm(nlp.pipe(docs, batch_size = 32), total = len(docs)):

        # Add each entity in doc to list
        entities = [(ent.text, ent.label_) for ent in doc.ents]
        
        # count each entity
        for ent in doc.ents:
            ent_counts.get(ent.label_).update([ent.text])
        
        # add list of entities to output
        ent_data.append(entities)
        
    return ent_data, ent_counts

def get_ent_counts(ent_data, ner_labels):
    '''
    Count number of entities extracted.

    Input:
        [ent_data]              | entities extracted from a list of documents
        [ner_labels]            | list of named entity labels

    Returns:
        [ent_counts]            | dictionary of counter objects.
                                | format:   {ner_label : {entity_1 : count, 
                                                          entity_2 : count}, ...
                                                         },
                                            ner_label_2 : {...}, ...
                                            }
    '''
    # initialize counters for each label
    ent_counts = {label : Counter() for label in ner_labels}

    # iterature through each list of extracted entities in ent_data
    for ents in ent_data:

        # update entity counts
        for ent in ents:
            txt, label = ent
            ent_counts.get(label).update([txt])
    
    return ent_counts

#########################################################
## Output manipulation functions ##

def select_ents(entsIn, label):
    '''
    Select entities with a given label. 

    Input:
        [entsIn]                | list of entities
        [label]                 | label you want to select

    Returns:
        [entsOut]               | list of entities with [label]

    '''
    entsOut = [(ent.text, ent.label_) for ent in entsIn if ent.label_ == label]
    return entsOut

def get_top_ent_counts(ent_counts, label, n = 10):
    '''
    Display top n counts for a given label category

    Input:
        [ent_counts]            | dictionary of counter objects.
        [label]                 | label you want to count
        [n]                     | number of entities to display

    Returns:
        [res]                   | list of entities and counts in descending order

    '''
    # check if label in count dictionary
    if label not in ent_counts.keys():
        print("Label not found.")
        print(f"Available labels: {list(ent_counts.keys())}")
        return
    
    # get top n entities
    res = [(ent, count) for ent, count in ent_counts[f"{label}"].most_common(n)]

    # return results
    return res

def find_entities(df : pd.DataFrame , ent : tuple[str, str]):
    '''
    Find a specific entity in 'haunted_places_features_addedv2.tab'

    Input:
        [df]                    | dataFrame with "Named_Entities" column
        [ent]                   | desired (entity, label pair)

    Returns:
        [idxs]                  | list of indices that contain [ent]

    '''
    # check column
    if "Named_Entities" not in df.columns:
        print("Column 'Named_Entities' not found.")
        return
    
    txt, label = ent

    # boolean masks for named entities that match
    mask = df['Named_Entities'].apply(lambda x: (txt,label) in x if isinstance(x, list) else False)
    idxs = df.loc[mask].index.tolist()
    # return indicies that match
    return idxs

def find_entities_by_label(df : pd.DataFrame , label : str):
    '''
    Find all haunted places with a specific label in 'haunted_places_features_addedv2.tab'

    Input:
        [df]                    | dataFrame with "Named_Entities" column
        [label]                 | label

    Returns:
        [idxs]                  | list of indices that contain [ent]
    '''
    # check if "Named_Entities" in dataframe
    if "Named_Entities" not in df.columns:
        print("Column 'Named_Entities' not found.")
        return
    
    # boolean masks for haunted places that contain an entity with [label]
    mask = df["Named_Entities"].apply(lambda row : any(map(lambda x: x[1] == label, row) if isinstance(row, list) else False))
    idxs = df.loc[mask].index.tolist()
    # return indicies that match
    return idxs

### **Feature Extraction**

In [ ]:
#########################################################
## Load Model and input data ##
nlp = sp.load("en_core_web_trf" , disable= [
    "parser",
    "attribute_ruler", 
    "lemmatizer"
])

# Display possible labels
ner_labels = nlp.get_pipe("ner").labels
print("Available labels: ", ner_labels)

docs = df.loc[:, "Description"].astype(str).tolist()

#########################################################
## Extract Named Entities  ##
ent_data, ent_counts = process_docs(docs)

df["Named_Entities"] = ent_data

#########################################################
## Save results ##
print("Saving to CSV...")

# Read Feature added dataframe
out_df = pd.read_csv(f"{outfile}", sep = "\t")


# If it exists, update values
if feature in out_df.columns:
    out_df[feature].update(df[feature].values)
    out_df[feature] = df[feature].values

# If not add entire column
else:
    out_df[feature] = df[feature].values

out_df.to_csv(f"{outfile}", sep = "\t", index = True)

print(f"CSV Saved to {outfile}")

### **SpaCY Performance**

What can SpaCY tell us about our haunted places?

---

#### Coverage
  - SpaCY extracted **34044** total entities.
  - **83.52%** of our haunted places had at least 1 named entity. 
  - **Date**, **Time**, and **FAC** had the highest coverage

 |          |   %_Coverage |   Extracted_Entities |
|:---------|-------------:|---------------------:|
| Total    |        83.52 |                34044 |
| DATE     |        39.52 |                 7168 |
| TIME     |        33.25 |                 4927 |
| FAC      |        26.72 |                 4474 |
| CARDINAL |        25.15 |                 4340 |
| PERSON   |        17.84 |                 3486 |
| ORG      |        15.16 |                 2341 |
| GPE      |        13.02 |                 2132 |

#### Uniqueness
  - SpaCy extracted **14921** unique entities total. Ratio of unique to total is **43.83\%** 

  - **Most unique entity types** - **FAC**, **DATE**, and **PERSON**. 
    - Makes intuitive sense, most people, places, and dates are unique.
  - **Dates are often repeated** - uniquness ratio of **33.27%**. **

 |        |   Unique_Entities |   Extracted_Entities |   %_Unique_Ratio |
|:-------|------------------:|---------------------:|-----------------:|
| Total  |             14921 |                34044 |            43.83 |
| FAC    |              3836 |                 4474 |            85.74 |
| DATE   |              2385 |                 7168 |            33.27 |
| PERSON |              2170 |                 3486 |            62.25 |
| ORG    |              1893 |                 2341 |            80.86 |
| GPE    |              1344 |                 2132 |            63.04 |
| TIME   |               833 |                 4927 |            16.91 |
| LOC    |               815 |                  935 |            87.17 |


#### Top Entities

 |    | PERSON            | PRODUCT                     | ORG                             | FAC                   | GPE                  | LOC                            | NORP                     | DATE                     | TIME                     |
|---:|:------------------|:----------------------------|:--------------------------------|:----------------------|:---------------------|:-------------------------------|:-------------------------|:-------------------------|:-------------------------|
|  0 | ('Mary', 64)      | ('Calvary', 4)              | ('Inn', 57)                     | ('Auditorium', 23)    | ('Chicago', 21)      | ('Gravity Hill', 7)            | ('Indian', 249)          | ('this day', 169)        | ('night', 1715)          |
|  1 | ('George', 49)    | ('911', 3)                  | ('Union', 30)                   | ('Cemetery', 15)      | ('Ohio', 20)         | ('Bandera Pass', 6)            | ('Indians', 122)         | ('today', 115)           | ('late at night', 499)   |
|  2 | ('Alice', 30)     | ('Zodiac', 3)               | ('KKK', 19)                     | ('Main Street', 14)   | ('California', 16)   | ('the Ohio River', 4)          | **('Confederate', 50)**      | ('years', 113)           | ('midnight', 208)        |
|  3 | ('Elizabeth', 28) | **('EverQuest', 2)**            | **('NASA', 16)**                    | ('Mansion', 12)       | ('Indiana', 15)      | ('Mississippi', 4)             | ('Native American', 44)  | ('the years', 110)       | ('the night', 198)       |
|  4 | ('John', 24)      | ('the Steel Phantom', 2)    | ('University', 13)              | ('Chapel', 11)        | ('Texas', 13)        | ('Slippery Rock Creek', 3)     | ('British', 28)          | ('the day', 102)         | ('one night', 155)       |
|  5 | ('Joe', 20)       | ('Colossus', 2)             | ('Wal-Mart', 11)                | ('Fort', 8)           | ('England', 12)      | ('an Indian Burial Ground', 3) | ('Spanish', 23)          | ("the early 1900's", 96) | ('nights', 114)          |
|  6 | ('Sarah', 18)     | ('The Goat Man', 2)         | **('YMCA', 11)**                    | ('Crybaby Bridge', 7) | ('Michigan', 11)     | ('Hills', 3)                   | ('Catholic', 20)         | ('years ago', 87)        | ('hours', 109)           |
|  7 | ('Charlie', 16)   | ('Impala', 2)               | ('Paramount', 11)               | ('Smith Hall', 7)     | ('US', 11)           | ('Beaver Creek', 3)            | **('confederate', 20)**      | ('many years ago', 86)   | ('Late at night', 96)    |
|  8 | ('Molly', 14)     | ('Roadrunner', 2)           | ('Hotel', 9)                    | ('Alumni Hall', 7)    | ('Pennsylvania', 10) | ('Lake Ontario', 3)            | ('Native Americans', 16) | ('many years', 85)       | ('One night', 71)        |
|  9 | ('Hannah', 14)    | ('HELP', 2)                 | ('Church', 9)                   | ("St. Mary's", 7)     | ('Illinois', 10)     | ('Island', 3)                  | ('Chinese', 16)          | ("the 1800's", 83)       | ('every night', 65)      |
| 10 | ('David', 12)     | ('H-1', 2)                  | ('the Underground Railroad', 7) | ('High School', 7)    | ('America', 9)       | ('South', 3)                   | ('French', 14)           | ('Halloween', 82)        | ('the morning', 58)      |
| 11 | **('Jesus', 12)**     | ('The Green Lady', 2)       | ('Academy', 6)                  | ('Main St.', 6)       | ('NY', 9)            | ('gravity hill', 3)            | ('German', 14)           | **('June 2008', 76)**        | ('morning', 34)          |
| 12 | ('Tommy', 12)     | ('the Greenbriar Light', 2) | ('Haunted Places', 6)           | ('Wilson Hall', 6)    | ('Missouri', 9)      | ('choate', 3)                  | ('Irish', 13)            | ("the late 1800's", 69)  | ('the next morning', 32) |
| 13 | ('Annie', 12)     | ('Torries', 2)              | ('State', 6)                    | ('Broadway', 6)       | ('Georgia', 9)       | ('Earth', 3)                   | ('Hawaiian', 13)         | **('March 2008', 58)**       | ('that night', 30)       |
| 14 | **('Al Capone', 11)**| ('Cheyenne', 2)             | ('Wal', 6)                      | ('Bridge', 6)         | ('Tennessee', 9)     | ('Lake', 3)                    | ('English', 12)          | ('the 1800', 54)         | ('dusk', 23)             |

#### :mag: Fun Observations
  - **Laws** — SpaCY identified two laws "ACT III
  - **Jesus** - "Jesus" is the 12th most *PERSON* common entity with 12 counts. 
    - "Al Capone" is close behind with 11 counts.
    - "Satan" only has 4 mentions
  - **EverQuest** - Guess we best stick to Runescape. 
  -  **YMCA** - It is not fun to stay in here. 
    - "Nasa" is also quite haunted. No mentions of JPL in our dataset though.
  - **Civil War** - Many mentions of "confederate". Looks like they lost the war.  
    - Only 30 mentions of "Union". 
  -  **Financial Crisis** - Most common dates are "June 2008" and "March 2008". 
    - Coincidence that this was during the financial crisis
  - **WORKS_OF_ARE** - "Bible" most common work of are with 11 mentions.
    - "West Side Story" comes in second with 3. 

---

#### 🛠 Improving upon `classify_time_of_day` from Assignment 1

SpaCY entities revealed critical oversights we made from Assignment 1. I improved the following from assignment 1:
- **DATE** - **`extract_dates`**
- **TIME** - **`classify_time_of_day`**

See sections below for more details. 


#### 📝 Conclusion and Connection to A1

In Conclusion SpaCy entities tell us a lot about our dataset. 

- **Storytelling** - More in depth labels like **PERSON** and **ORG** tell us what the important figures are across our dataset
- **Improved Parsing** - SpaCY entities revealed edge cases to our previous methods in A1. I document the changes in the codebook below.:
- **Visualize our Data** - Histograms below are a cool way to visualize what author's believe are the most relevant entities for a haunted place. 
  - **DATE** , **TIME**, **FAC** are the most relevant to a haunted place. 


In [ ]:
#########################################################
## count number of entities extracted ##
ent_counts = get_ent_counts(df["Named_Entities"].values, ner_labels)

#########################################################
## Init output dict ##

analysisDict = {label: {} for label in ner_labels}

for label in ner_labels:
    # Number of unique entities
    analysisDict[label]["Unique_Entities"] = len(ent_counts[label].keys())

    # Number of extracted entities
    analysisDict[label]["Extracted_Entities"] = sum(ent_counts[label].values())

    # Uniqueness ratio
    analysisDict[label]["%_Unique_Ratio"] = round(analysisDict[label]["Unique_Entities"] / analysisDict[label]["Extracted_Entities"] * 100, 2)

    # Label coverage across df
    analysisDict[label]["%_Coverage"] =  round((len(find_entities_by_label(df, label))) / df.shape[0] * 100 ,2)


rows_with_any_entity = df["Named_Entities"].apply(lambda row : any(map(lambda x: x[1] == label, row) if isinstance(row, list) else False))

#########################################################

## Dataframe and total statistics ##
dfTemp = pd.DataFrame.from_dict(analysisDict, orient = "index")
dfTemp.loc["Total"] = [np.nan] * dfTemp.shape[1]

# Totals accross all labels 
dfTemp.loc["Total", "Unique_Entities"] = dfTemp["Unique_Entities"].sum()
dfTemp.loc["Total", "Extracted_Entities"] = dfTemp["Extracted_Entities"].sum()
dfTemp.loc["Total", "%_Unique_Ratio"] = round(dfTemp["Unique_Entities"].sum() / dfTemp["Extracted_Entities"].sum() * 100, 2) 

# Coverage across all labels
rows_with_any_entity = df["Named_Entities"].apply(lambda row : row != [] if isinstance(row, list) else False)
dfTemp.loc["Total", "%_Coverage"] =  round((sum(rows_with_any_entity) / df.shape[0]) * 100, 2)

dfTemp.sort_values(by = "Extracted_Entities", ascending = False)

In [ ]:
cols = ["Extracted_Entities", "Unique_Entities"]
idxs = dfTemp.index.to_list()
idxs.remove("Total") 

dfTemp

df_plot = (dfTemp.loc[idxs, cols]
           .sort_values(by = "Extracted_Entities", ascending = True)
           .reset_index()
           .melt(id_vars='index', value_vars=cols, var_name='Entity_Type', value_name='Count')
)

df_plot.rename(columns={'index': 'Label'}, inplace=True)

plt.figure(figsize=(10, 6))
sns.set(style='whitegrid', font='Times New Roman')
sns.barplot(data=df_plot, x='Label', y='Count', hue='Entity_Type')


plt.title(f'Entity Counts', fontsize=16)
plt.xlabel('Label', fontsize = 14)
plt.ylabel('Count', fontsize = 14)
plt.xticks(rotation = 45, ha = 'right')
plt.tight_layout()
plt.show()

In [ ]:
cols = ["%_Unique_Ratio"]
idxs = dfTemp.index.to_list()
idxs.remove("Total") 

dfTemp

df_plot = (dfTemp.loc[idxs, cols]
           .sort_values(by = "%_Unique_Ratio", ascending = True)
           .reset_index()
           .melt(id_vars='index', value_vars=cols, var_name='Entity_Type', value_name='Count')
)

df_plot.rename(columns={'index': 'Label'}, inplace=True)

plt.figure(figsize=(10, 6))
sns.set(style='whitegrid', font='Times New Roman')
sns.barplot(data=df_plot, x='Label', y='Count', hue='Label', palette = "ch:s=.5,rot=-.5")


plt.title(f'Uniqueness Ratio', fontsize=16)
plt.xlabel('Label', fontsize = 14)
plt.ylabel('Percent', fontsize = 14)
plt.xticks(rotation = 45, ha = 'right')
plt.tight_layout()
plt.show()

In [ ]:
cols = ["%_Coverage"]
idxs = dfTemp.index.to_list()
idxs.remove("Total") 

dfTemp

df_plot = (dfTemp.loc[idxs, cols]
           .sort_values(by = "%_Coverage", ascending = True)
           .reset_index()
           .melt(id_vars='index', value_vars=cols, var_name='Entity_Type', value_name='Count')
)

df_plot.rename(columns={'index': 'Label'}, inplace=True)

plt.figure(figsize=(10, 6))
sns.set(style='whitegrid', font='Times New Roman')
sns.barplot(data=df_plot, x='Label', y='Count', hue='Label', palette = "ch:s=.5,rot=-.5")


plt.title(f'Percent Coverage', fontsize=16)
plt.xlabel('Label', fontsize = 14)
plt.ylabel('Percent', fontsize = 14)
plt.xticks(rotation = 45, ha = 'right')
plt.tight_layout()
plt.show()

### **Improving on Assignment 1**

What new information do we gain from these labels? I analyzed the labels I thought were the most important below.

#### **DATE**

How do the **"DATE"** entities improve upon our existing **"Haunted_Places_Date"** feature? 

---

##### 🧭 Adding Context to Haunted Places
- SpaCy recognizes the following:
  - **Seasons** — e.g., `("Winter", 13)`, `("Summer", 55)`
  - **Holidays** — e.g., `("Halloween", 82)`, `("Christmas", 14)`
  - **Individual Months** — e.g., `("December", 13)`, `("October", 18)`
  - **Relative Dates** — e.g., `("this day", 169)`, `("Many years ago", 33)`

---

##### 🔧 Improving `extract_dates` from Assignment 1

- **extract_dates from assignment 1** only captured **45%** of the top 100 most common "DATE" entities. Of these entities, we missed:
   

  | Missed Entity      | Enhancement                                                                 |
  |--------------------|------------------------------------------------------------------------------|
  | **"1800s"**         | Updated regex to allow optional "s": `(?:\s*'?s)?`                          |
  | **Double Years**    | Modified `clean_dates` to remove duplicate years                            |
  | **"20th Century"**  | Added regex: `\b\d{1,2}(?:st|nd|rd|th)?\s*century\b`                         |
  | **Holidays**        | Introduced regex patterns using `Holiday_patterns` dictionary at function top |
  |                    | Holidays stored as `[1000, [month], [day]]`                                  |

---

All of these improvements were compiled into the new **`extract_dates`** function, located in:

```
./dsci_550_a1/parsingFunctions_v2.py
```

##### 📝 Conclusion

  | Method      | Coverage of top 100 entities | Total Coverage |                                          
  |--------------------|----------------------------------------|--------------------------------------|
  | **extract_dates**         | 43%                         |   23%
  | **extract_dates_v2**    | **67%**                           |   **41.17%**
                        |


In Conclusion SpaCy the "DATE" entities:
- **revealed edge cases**: 
    - After improvements, `extract_dates` captures **67%** of the top 100 most common "DATE" entities
    - Overall, `extract_dates` now covers **41.17%** of all "DATE" entities
- **contextualize the timeline of a haunted place**: 
    - Entities like "the next day" provide relative temporal information.

In [ ]:
## Get top counts ##
label = "DATE"
n = 100
res = get_top_ent_counts(ent_counts, label, n = n)

# Initialize counters for dates parsable by 'extract_dates' function from assignment 1 
total_parsable_by_A1 = 0

#########################################################
## Table Generation ##
header = f'| {"Entity":^25}|{"Count":^25}|{"Parsed Dates":^25}|'
table_length = len(header)
rows = []

# loop through top counts 
for ent, count in res:
    parsed_dates = []

    # Check if date is parsable by extract_dates function from assignment 1:
    for val in extract_dates(ent).values():
        if isinstance(val, list):
            [parsed_dates.append(v) for v in val if v != datetime.date(2025, 1, 1)]
    
    # If parsed_dates is not empty. We parsed the correct date. Increment count
    if len(parsed_dates) != 0:
        total_parsable_by_A1 += 1

    # Remove duplicate before printout
    clean_dates(parsed_dates)

    ## |print entity | count | parsed dates| ##
    rows.append(f'| {ent:^25}|{count:^25}|{str(parsed_dates).replace("datetime.date",""):^25}|')

#########################################################

## Total % coverage of parser from A1 accross top 100 entities ##
total_coverage_top_n = round((total_parsable_by_A1 / n) * 100, 2)

## Total % coverage of parser from A1 accross all entities ##
total_parsable_by_A1 = 0 

for ent in ent_counts[label].keys():
    parsed_dates = []

    for val in extract_dates(ent).values():
        if isinstance(val, list):
            [parsed_dates.append(v) for v in val if v != datetime.date(2025, 1, 1)]
    
    if len(parsed_dates) != 0:
        total_parsable_by_A1 += 1

total_coverage = round((total_parsable_by_A1 / len(ent_counts[label].keys())) * 100, 2)

print("-" * table_length)
print(f"Of the top {n} '{label}' entities, 'extract_dates_v1' could parse : {43:2g}%")
print(f"Now 'extract_dates_v2' can parse : {total_coverage_top_n:2g}%")

print(f"Across all '{label}' entities, 'extract_dates_v1' could parse : {24:2g}%")
print(f"Now 'extract_dates_v2' can parse : {total_coverage:2g}%")
print("-" * table_length)

print(f"Top {n} Counts for {label}:".center(table_length))
print("-" * table_length)
print(header)
print("-" * table_length)
[print(r) for r in rows]
print("-" * table_length)


#### **TIME**

**How does _"TIME"_ improve upon _Time_of_Day_?**

---

##### 🏚️ Added Context to Haunted Place

- Named entities provide context for the **duration of events** that _Time_of_Day_ does not:
  - **"a few minutes later"**
  - **"the next day"**
  - **"seconds before"**
- These features don't fit into the limited categories of _Time_of_Day_, but help provide **temporal context** to narratives.

---

##### 🛠 Improving upon `classify_time_of_day` from Assignment 1

- **`classify_time_of_day` from Assignment 1:**
  - Captured **57%** of the top 100 most common "DATE" entities
  - Captured only **36.37%** of all "DATE" entities overall

- Here are some missed entities and how we addressed them:

  | Missed Entities                            | Fix / Enhancement                                                                 |
  |--------------------------------------------|------------------------------------------------------------------------------------|
  | **"nights", "evenings", "mornings"**       | Added suffix regex: `('s|s)?`                                                     |
  | **"overnight", "morningtime"**             | Used more general regular expressions with lemmatization                          |
  | **"Afternoon", "All Hours", "all day"**    | Added new feature values: **"Afternoon"**, **"All Day"**                          |
  | **"3 pm"**                                  | Created `classify_hours` function to parse specific times of day                  |
  | **"school hours", "closing hours"**        | Used regex: `\b(?:school|business|work|day|lunch|dinner|daytime)\s*hours?\b`     |

---

All of these improvements were compiled into the new **`classify_time_of_day_v2`** function, located in:

```
./dsci_550_a2/parsingFunctions_v2.py
```

In [ ]:
#########################################################
# A2 and A1 classify_time_of_day_v2 functions #
from dsci_550_a2.parsingFunctions_v2 import classify_time_of_day_v2, classify_hours

time_patterns = {
    "Morning": r"\bmorning\b|\bdawn\b|\bsunrise\b",
    "Evening": r"\bevening\b|\bnight\b|\bmidnight\b|\blate\b",
    "Dusk": r"\bdusk\b|\bsunset\b|\btwilight\b"
}
def classify_time_of_day(text):
    if not isinstance(text, str):
        return "Unknown"

    text = text.lower()
    
    for label, pattern in time_patterns.items():
        if re.search(pattern, text):
            return label  # Return first match found

    return "Unknown"

#########################################################
## Get top counts ##
label = "TIME"
n = 100
res = get_top_ent_counts(ent_counts, label, n = n)

# Initialize counters for dates parsable by 'extract_dates' function from assignment 1 and assignment 2
total_parsable_by_A1 = 0
total_parsable_by_A2 = 0

#########################################################
## Table Generation ##
column_width = max(len(ent) for ent, _ in res)
header = f'| {"Entity":^{column_width}}|{"Count":^{column_width}}|{"Parsed Dates_v1":^{column_width}}|{"Parsed Dates_v2":^{column_width}}|'
table_length = len(header)
rows = []

# loop through top counts 
for ent, count in res:
    parsed_dates = []

    # Check if date is parsable by classify_time_of_day function from assignment 1 and assignment 2:
    a1_result, a2_result = classify_time_of_day(ent), classify_time_of_day_v2(ent)
    total_parsable_by_A1 += a1_result != "Unknown"
    total_parsable_by_A2 += a2_result != "Unknown"
    

    ## |print entity | count | parsed dates| ##
    rows.append((f'| {ent:^{column_width}}|{count:^{column_width}}|{a1_result:^{column_width}}|{a2_result:^{column_width}}|'))

#########################################################
## Total % coverage of both parsers accross top 100 "TIME" ##
total_coverage_a1_top_n = round((total_parsable_by_A1 / n) * 100, 2)
total_coverage_a2_top_n = round((total_parsable_by_A2 / n) * 100, 2)



#########################################################
## Total % coverage of parser from both accross all entities ##
total_parsable_by_A1 = 0 

# loop through top counts 
for ent in ent_counts[label].keys():
    parsed_dates = []

    # Check if date is parsable by classify_time_of_day function from assignment 1 and assignment 2:
    a1_result, a2_result = classify_time_of_day(ent), classify_time_of_day_v2(ent)
    total_parsable_by_A1 += a1_result != "Unknown"
    total_parsable_by_A2 += a2_result != "Unknown"

# Calculate total coverage for both
total_coverage_a1 = round((total_parsable_by_A1 / len(ent_counts[label].keys())) * 100, 2)
total_coverage_a2 = round((total_parsable_by_A2 / len(ent_counts[label].keys())) * 100, 2)


print("-" * table_length)
print(f"Of the top {n} '{label}' entities, 'classify_time_of_day' can parse : {total_coverage_a1_top_n:2g}%")
print(f"Of the top {n} '{label}' entities, 'classify_time_of_day_v2' can parse : {total_coverage_a2_top_n:2g}%")
print("\n")
print(f"Across all '{label}' entities, 'classify_time_of_day' can parse : {total_coverage_a1:2g}%")
print(f"Across all '{label}' entities, 'classify_time_of_day_v2' can parse : {total_coverage_a2:2g}%")
print("-" * table_length)


print(f"Top {n} Counts for {label}:".center(table_length))
print("-" * table_length)
print(header)
print("-" * table_length)
[print(r) for r in rows]
print("-" * table_length)


### **Regenerated A1 Features With Improved Functions**

Applying the same methodology of notebooks `1.03` and `1.07`, we saw the following improvements using the updated scripts

|    |   Time_of_Day |   Haunted_Places_Date |
|:---|--------------:|----------------------:|
| v1 |       32.84\% |               25.48\% |
| v2 |       **35.34**\% |               **27.57**\% |


In [10]:
from dsci_550_a2.parsingFunctions_v2 import classify_time_of_day_v2
from dsci_550_a1.parsingFunctions import extract_dates

# Output Df
outfile = "../data/processed/haunted_places_features_added_v2.tab"

# Reading CSV
df_input = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep = "\t")
df_raw = pd.read_csv("../data/raw/haunted_places.tab", sep = "\t")
df_comparison = pd.read_csv("../data/processed/haunted_places_features_added.tab", sep = "\t")


## Time of Day with new and improved scripts##
df["Time_of_Day"] = df["Description"].apply(classify_time_of_day_v2)

## Extract Dates with new and improved scripts ##
df["Haunted_Places_Date_Cleaned"] = df["Description"].apply(extract_dates)

df["Haunted_Places_Date_Raw"] = df_raw["description"].apply(extract_dates)

# Final feature column
df["Haunted_Places_Date"] = (df
    # Combine cleaned and raw
    .apply(lambda x: x['Haunted_Places_Date_Cleaned']['dates'] +  x['Haunted_Places_Date_Raw']['dates'], axis = 1)
    # Remove Duplicates
    .apply(lambda x: list(set(x)))
    # remove [2025, 1, 1] if there are any valid dates
    .apply(lambda x: clean_dates(x))
)

/Users/dgottschalk/miniconda3/envs/dsci_550_a1/lib/python3.12/site-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname M identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


In [30]:
dict_md = {
    "v1" : [],
    "v2" : []
}
n = df_comparison.shape[0]
dict_md["v1"] = [(1 - df_comparison["Time_of_Day"].value_counts()["Unknown"] / n) * 100, (1 - df_comparison["Haunted_Places_Date"].value_counts()[0] / n) * 100]
dict_md["v2"] = [(1 - df["Time_of_Day"].value_counts()["Unknown"] / n) * 100, (1 - df["Haunted_Places_Date"].value_counts()[0] / n) * 100]

df_md = pd.DataFrame.from_dict(dict_md, orient = "index", columns = ["Time_of_Day", "Haunted_Places_Date"])
print(df_md.to_markdown(index = True))

|    |   Time_of_Day |   Haunted_Places_Date |
|:---|--------------:|----------------------:|
| v1 |       32.8421 |               25.4822 |
| v2 |       35.3439 |               27.5655 |


/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_48571/3534672550.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dict_md["v1"] = [(1 - df_comparison["Time_of_Day"].value_counts()["Unknown"] / n) * 100, (1 - df_comparison["Haunted_Places_Date"].value_counts()[0] / n) * 100]
/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_48571/3534672550.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dict_md["v2"] = [(1 - df["Time_of_Day"].value_counts()["Unknown"] / n) * 100, (1 - df["Haunted_Places_Date"].value_counts()[0] / n) * 100]


In [ ]:

# cleaned data
df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep = "\t")
# raw data
df_raw = pd.read_csv("../data/raw/haunted_places.tab", sep = "\t")

feature_names = ["Haunted_Places_Date"]


start = time.time()
# Parse cleaned dataset
df["Haunted_Places_Date_Cleaned"] = df["Description"].apply(extract_dates)

# Parse raw dataset
df["Haunted_Places_Date_Raw"] = df_raw["description"].apply(extract_dates)

# Final feature column
df["Haunted_Places_Date"] = (df
    # Combine cleaned and raw
    .apply(lambda x: x['Haunted_Places_Date_Cleaned']['dates'] +  x['Haunted_Places_Date_Raw']['dates'], axis = 1)
    # Remove Duplicates
    .apply(lambda x: list(set(x)))
    # remove [2025, 1, 1] if there are any valid dates
    .apply(lambda x: clean_dates(x))
)



In [31]:
## Save CSV ##
print("Saving to CSV...")

# Read Feature added dataframe
out_df = pd.read_csv(f"{outfile}", sep = "\t")


# Check if feature exists
for feature in ["Haunted_Places_Date", "Time_of_Day"]:

    # If it exists, update values
    if feature in out_df.columns:
        out_df[feature].update(df[feature].values)
        out_df[feature] = df[feature].values

    # If not add entire column
    else:
        out_df[feature] = df[feature].values

out_df.to_csv(f"{outfile}", sep = "\t", index = False)

print(f"CSV Saved to {outfile}")

Saving to CSV...


/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_48571/3597128958.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  out_df[feature].update(df[feature].values)


CSV Saved to ../data/processed/haunted_places_features_added_v2.tab
